# FIT4112 – Lab 3 | Notebook 03: Customer Segmentation Insights

**Mục tiêu chung:** Không dừng lại ở `fit_predict`. Hãy biến nhãn cụm thành **mô tả khách hàng có thể hiểu được** và đề xuất hành động.

**Nguyên tắc:** Đặt tên cụm dựa trên **giá trị RFM gốc**, không dựa vào số cluster hay tọa độ đã chuẩn hóa.

**Đầu vào:** `data/processed/customer_segments.csv` do Notebook 02 sinh ra.

## Bước 1. Đọc kết quả phân cụm

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def find_project_root():
    current = Path.cwd().resolve()
    for directory in [current, *current.parents]:
        if (directory / "data" / "raw" / "retail_transactions_demo.csv").exists():
            return directory
    raise FileNotFoundError("Không tìm thấy repo FIT4112 Lab 3. Mở notebook từ repo hoặc notebooks/.")

ROOT = find_project_root()
RAW_DIR = ROOT / "data" / "raw"
OUT_DIR = ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Thư mục dự án:", ROOT)

path = OUT_DIR / "customer_segments.csv"
if not path.exists():
    raise FileNotFoundError("Chạy Notebook 02 trước để tạo customer_segments.csv")
segments = pd.read_csv(path, dtype={"CustomerID": "string"})
expected = {"CustomerID", "Recency", "Frequency", "Monetary", "Cluster"}
assert expected.issubset(segments.columns)
print("Khách hàng:", len(segments), "| Số cụm:", segments["Cluster"].nunique())
display(segments.head())

## Bước 2. Thống kê profile cụm

Dùng trung vị (median) để giảm ảnh hưởng của khách hàng có chi tiêu quá lớn. Có thể dùng thêm trung bình sau khi quan sát dữ liệu.

In [ ]:
profile = (
    segments.groupby("Cluster")
    .agg(
        Customers=("CustomerID", "nunique"),
        Recency_median=("Recency", "median"),
        Frequency_median=("Frequency", "median"),
        Monetary_median=("Monetary", "median"),
    )
    .reset_index()
    .sort_values("Cluster")
)
profile["Share_pct"] = profile["Customers"] / len(segments) * 100
display(profile.round(2))
assert profile["Customers"].sum() == len(segments)

## Bước 3. Đặt tên cụm (gợi ý, không phải đáp án duy nhất)

Dưới đây là **gợi ý tự động theo tương quan giữa các trung vị cụm**. Bạn phải kiểm tra bảng thực tế và **chỉnh tên nếu thấy không hợp lý**.

Lưu ý: Recency nhỏ nghĩa là khách hàng mua gần đây; Monetary cao nghĩa là tổng chi tiêu cao.

In [ ]:
median_rec = profile["Recency_median"].median()
median_mon = profile["Monetary_median"].median()
def suggested_name(row):
    active = row["Recency_median"] <= median_rec
    high_value = row["Monetary_median"] >= median_mon
    if active and high_value:
        return "Gần đây – giá trị cao"
    if active and not high_value:
        return "Gần đây – giá trị thấp"
    if not active and high_value:
        return "Lâu chưa mua – giá trị cao"
    return "Lâu chưa mua – giá trị thấp"

profile["SuggestedName"] = profile.apply(suggested_name, axis=1)
display(profile.round(2))
profile.to_csv(OUT_DIR / "segment_profiles.csv", index=False)
print("Đã lưu segment_profiles.csv")

## Bước 4. So sánh hai chỉ số RFM giữa các cụm

Đặt các nhóm cạnh nhau để lựa chọn ưu tiên chăm sóc khách hàng. Không kết luận dựa vào màu sắc hoặc số hiệu Cluster.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
profile.plot.bar(x="Cluster", y="Monetary_median", ax=axes[0], legend=False, title="Chi tiêu trung vị")
profile.plot.bar(x="Cluster", y="Recency_median", ax=axes[1], legend=False, title="Số ngày kể từ lần mua gần nhất")
axes[0].set_ylabel("Giá trị tiền tệ (đơn vị demo)")
axes[1].set_ylabel("Ngày")
plt.tight_layout()
plt.show()

## Bước 5. Phân tích và đưa ra quyết định

**TODO 1:** Dựa trên `profile`, đặt tên **tất cả cụm** theo số liệu thực tế. Với k = 2, mô tả đủ 2 nhóm; với k ≥ 3, mô tả ít nhất 3 nhóm và có thể bổ sung các nhóm khác.

**TODO 2:** Đề xuất **2 hành động kinh doanh** cho 2 phân khúc khác nhau. Ví dụ: chăm sóc nhóm thường mua và chi tiêu cao; chiến dịch tái kích hoạt nhóm lâu chưa quay lại. Với **mỗi hành động**, ghi cụm liên quan và số liệu RFM làm bằng chứng.

**TODO 3:** Viết **ít nhất một hạn chế**. Dữ liệu demo là giả lập, silhouette không phản ánh trực tiếp hiệu quả chiến dịch, cluster label không biểu diễn cấp bậc và khách hàng có thể đổi cụm theo thời gian.

**Foundation:** ưu tiên trả lời câu hỏi và liên hệ kinh doanh.

**Applied:** giải thích thêm vì sao cấu hình đặc trưng/số k được chọn tạo ra những nhóm có thể hành động, và khi nào doanh nghiệp nên đánh giá lại.

Nộp 3 notebook và báo cáo ngắn theo `docs/REPORT_TEMPLATE.md`.